In [1]:
import os
import random
import pandas as pd
from sklearn.model_selection import train_test_split

# Make runs reproducible
random.seed(42)

OUTPUT_DIR = "data"
TRAIN_PATH = os.path.join(OUTPUT_DIR, "prompts_train.csv")
VAL_PATH = os.path.join(OUTPUT_DIR, "prompts_val.csv")

NUM_SAMPLES = 10_000

# -----------------------------
# Category definitions
# -----------------------------

def gen_coding_prompt():
    langs = ["Python", "JavaScript", "Java", "C++", "Go"]
    tasks = [
        "sort a list of numbers",
        "read a file and count word frequency",
        "implement a binary search",
        "build a REST API endpoint",
        "validate user input",
    ]
    extras = [
        "explain the solution step-by-step",
        "add inline comments",
        "provide time and space complexity",
        "include at least 5 test cases",
        "optimize for readability",
    ]
    lang = random.choice(langs)
    task = random.choice(tasks)
    extra = random.sample(extras, k=random.randint(1, 3))

    raw = (
        f"hey chatgpt can you write some {lang} code to {task} and also "
        f"{' and '.join(extra)} but keep it simple pls"
    )

    # Optimized version
    lines = [
        f"Write a {lang} program to {task}.",
        "Requirements:",
    ]
    for e in extra:
        lines.append(f"- {e[0].upper()}{e[1:]}.")
    optimized = "\n".join(lines)

    return raw, optimized


def gen_writing_prompt():
    tones = ["professional", "friendly", "simple", "formal", "casual"]
    actions = ["rewrite", "improve", "shorten", "expand", "polish"]
    tone = random.choice(tones)
    action = random.choice(actions)

    raw = (
        f"pls {action} this paragraph and make it sound {tone} but not too complex "
        f"and make sure there are no grammar mistakes and keep meaning same"
    )

    optimized = (
        f"{action.capitalize()} the paragraph.\n"
        f"- Tone: {tone}.\n"
        "- Correct grammar.\n"
        "- Preserve the original meaning.\n"
        "- Avoid unnecessary complexity."
    )
    return raw, optimized


def gen_reasoning_prompt():
    topics = [
        "dynamic programming",
        "Bayes theorem",
        "binary search trees",
        "time complexity in algorithms",
        "gradient descent",
    ]
    topic = random.choice(topics)
    raw = (
        f"can you explain {topic} but like slowly and step by step because i get confused "
        "and maybe show your thinking process or something"
    )
    optimized = (
        f"Explain {topic} step-by-step.\n"
        "- Use simple language.\n"
        "- Show the reasoning process clearly.\n"
        "- Include a small example."
    )
    return raw, optimized


def gen_role_prompt():
    roles = [
        "career coach",
        "senior Python developer",
        "English grammar tutor",
        "data science mentor",
        "interviewer for software engineering",
    ]
    tasks = [
        "review my answers and give feedback",
        "ask me questions one by one",
        "help me learn key concepts",
        "suggest improvements",
    ]
    role = random.choice(roles)
    task = random.choice(tasks)

    raw = f"pls act as {role} and {task} but not too strict and keep things simple and short"
    optimized = (
        f"Act as a {role}.\n"
        f"Task: {task}.\n"
        "- Keep responses simple and concise.\n"
        "- Maintain a supportive tone."
    )
    return raw, optimized


def gen_exam_prompt():
    subjects = ["data structures", "operating systems", "DBMS", "machine learning", "computer networks"]
    levels = ["easy", "medium", "hard"]
    subject = random.choice(subjects)
    level = random.choice(levels)

    raw = (
        f"create some {subject} questions for exam prep maybe {level} level "
        "and also give answers but dont make them too long"
    )
    optimized = (
        f"Create {subject} exam questions.\n"
        f"- Difficulty: {level}.\n"
        "- Provide concise answers.\n"
        "- Avoid overly long explanations."
    )
    return raw, optimized


def gen_chatbot_system_prompt():
    styles = ["professional", "friendly", "technical", "beginner-friendly"]
    style = random.choice(styles)

    raw = (
        f"you are chatgpt please behave very {style} and answer my questions but also "
        "dont be too long and dont say you are an ai all the time"
    )
    optimized = (
        "System prompt for chatbot:\n"
        f"- Tone: {style}.\n"
        "- Provide clear and helpful answers.\n"
        "- Keep responses concise.\n"
        "- Do not repeatedly mention being an AI."
    )
    return raw, optimized


def gen_multilingual_prompt():
    # Very simple synthetic examples (you can extend with real multilingual data)
    languages = [
        ("tamil", "தமிழில் எளிமையாக எழுதவும்."),
        ("hindi", "हिंदी में सरल और स्पष्ट लिखें।"),
        ("english", "write in simple and clear English."),
    ]
    lang_name, instruction = random.choice(languages)
    raw = f"pls explain binary search and {instruction} and dont use heavy technical words"
    optimized = (
        f"Explain binary search.\n"
        f"- Language: {lang_name}.\n"
        f"- {instruction}\n"
        "- Avoid heavy technical terms."
    )
    return raw, optimized


GENERATORS = [
    gen_coding_prompt,
    gen_writing_prompt,
    gen_reasoning_prompt,
    gen_role_prompt,
    gen_exam_prompt,
    gen_chatbot_system_prompt,
    gen_multilingual_prompt,
]


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    rows = []
    for i in range(NUM_SAMPLES):
        gen = random.choice(GENERATORS)
        raw, optimized = gen()
        rows.append({
            "raw_prompt": raw,
            "optimized_prompt": optimized,
        })

    df = pd.DataFrame(rows)

    train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

    train_df.to_csv(TRAIN_PATH, index=False, encoding="utf-8")
    val_df.to_csv(VAL_PATH, index=False, encoding="utf-8")

    print(f"Saved {len(train_df)} train samples to {TRAIN_PATH}")
    print(f"Saved {len(val_df)} val samples to {VAL_PATH}")


if __name__ == "__main__":
    main()


Saved 9000 train samples to data\prompts_train.csv
Saved 1000 val samples to data\prompts_val.csv


In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np

# -------------------------------
# CONFIG
# -------------------------------
TRAIN_PATH = "data/prompts_train.csv"
VAL_PATH = "data/prompts_val.csv"
EMBED = 256
HIDDEN = 512
BATCH = 16
EPOCHS = 5
MAX_LEN = 80
MODEL_PATH = "tf_prompt_optimizer"

# -------------------------------
# SIMPLE TOKENIZER
# -------------------------------
class SimpleTokenizer:
    def __init__(self):
        self.tok = tf.keras.preprocessing.text.Tokenizer(filters="", oov_token="<unk>")
        
    def build(self, texts):
        self.tok.fit_on_texts(texts)
        self.tok.word_index["<pad>"] = 0
        
    @property
    def vocab(self):
        return len(self.tok.word_index) + 1
    
    def encode(self, text):
        seq = self.tok.texts_to_sequences([text])[0]
        return [1] + seq + [2]   # <bos>=1 <eos>=2
    
    def decode(self, ids):
        inv = {v:k for k,v in self.tok.word_index.items()}
        return " ".join(inv.get(i,"") for i in ids if i > 2)

# -------------------------------
# LOAD DATA
# -------------------------------
df_t = pd.read_csv(TRAIN_PATH)
df_v = pd.read_csv(VAL_PATH)

all_text = df_t["raw_prompt"].tolist() + df_t["optimized_prompt"].tolist()

tok = SimpleTokenizer()
tok.build(all_text)
VOCAB = tok.vocab

def encode_pad(text):
    seq = tok.encode(text)
    return seq[:MAX_LEN] if len(seq) > MAX_LEN else seq

X_train = [encode_pad(t) for t in df_t["raw_prompt"]]
Y_train = [encode_pad(t) for t in df_t["optimized_prompt"]]

X_val = [encode_pad(t) for t in df_v["raw_prompt"]]
Y_val = [encode_pad(t) for t in df_v["optimized_prompt"]]

X_train = tf.keras.preprocessing.sequence.pad_sequences(X_train, padding="post")
Y_train = tf.keras.preprocessing.sequence.pad_sequences(Y_train, padding="post")

X_val = tf.keras.preprocessing.sequence.pad_sequences(X_val, padding="post")
Y_val = tf.keras.preprocessing.sequence.pad_sequences(Y_val, padding="post")

train_ds = tf.data.Dataset.from_tensor_slices((X_train, Y_train)).shuffle(10000).batch(BATCH)
val_ds = tf.data.Dataset.from_tensor_slices((X_val, Y_val)).batch(BATCH)

# -------------------------------
# ATTENTION
# -------------------------------
class BahdanauAttention(layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.W1 = layers.Dense(units)
        self.W2 = layers.Dense(units)
        self.V = layers.Dense(1)

    def call(self, hidden, enc_out):
        hidden = tf.expand_dims(hidden, 1)
        score = self.V(tf.nn.tanh(self.W1(enc_out) + self.W2(hidden)))
        weights = tf.nn.softmax(score, axis=1)
        ctx = tf.reduce_sum(weights * enc_out, axis=1)
        return ctx, weights

# -------------------------------
# ENCODER
# -------------------------------
class Encoder(tf.keras.Model):
    def __init__(self, vocab, embed, hidden):
        super().__init__()
        self.embed = layers.Embedding(vocab, embed)
        self.lstm = layers.LSTM(hidden, return_sequences=True, return_state=True)

    def call(self, x):
        x = self.embed(x)
        out, h, c = self.lstm(x)
        return out, h, c

# -------------------------------
# DECODER
# -------------------------------
class Decoder(tf.keras.Model):
    def __init__(self, vocab, embed, hidden):
        super().__init__()
        self.embed = layers.Embedding(vocab, embed)
        self.lstm = layers.LSTM(hidden, return_sequences=True, return_state=True)
        self.attn = BahdanauAttention(hidden)
        self.fc = layers.Dense(vocab)

    def call(self, x, hidden, cell, enc_out):
        x = self.embed(x)
        ctx, _ = self.attn(hidden, enc_out)
        ctx = tf.expand_dims(ctx, 1)
        x = tf.concat([ctx, x], axis=-1)

        out, h, c = self.lstm(x, initial_state=[hidden, cell])
        logits = self.fc(out)
        return logits, h, c

# -------------------------------
# BUILD MODEL
# -------------------------------
encoder = Encoder(VOCAB, EMBED, HIDDEN)
decoder = Decoder(VOCAB, EMBED, HIDDEN)
optimizer = tf.keras.optimizers.Adam()

# -------------------------------
# TRAIN STEP
# -------------------------------
@tf.function
def train_step(src, tgt):
    loss = 0.0

    with tf.GradientTape() as tape:
        enc_out, h, c = encoder(src)
        dec_input = tf.expand_dims(tgt[:,0], 1)

        for t in range(1, tgt.shape[1]):
            logits, h, c = decoder(dec_input, h, c, enc_out)
            loss += tf.reduce_mean(
                tf.keras.losses.sparse_categorical_crossentropy(
                    tgt[:, t], logits[:,0], from_logits=True
                )
            )
            dec_input = tf.expand_dims(tgt[:, t], 1)

    vars = encoder.trainable_variables + decoder.trainable_variables
    grads = tape.gradient(loss, vars)
    optimizer.apply_gradients(zip(grads, vars))

    return loss / tf.cast(tgt.shape[1], tf.float32)

# -------------------------------
# TRAIN LOOP
# -------------------------------
for epoch in range(EPOCHS):
    total = 0
    steps = 0
    for src, tgt in train_ds:
        batch_loss = train_step(src, tgt)
        total += batch_loss
        steps += 1
    print(f"Epoch {epoch+1} Loss: {total/steps}")




Epoch 1 Loss: 0.5398494601249695
Epoch 2 Loss: 0.05869385227560997
Epoch 3 Loss: 0.034549321979284286
Epoch 4 Loss: 0.02384125255048275
Epoch 5 Loss: 0.017115086317062378


ValueError: The filename must end in `.weights.h5`. Received: filepath=tf_prompt_optimizer_enc

In [9]:
# -------------------------------
# SAVE
# -------------------------------
encoder.save_weights(MODEL_PATH + "_enc.weights.h5")
decoder.save_weights(MODEL_PATH + "_dec.weights.h5")
print("Model saved.")


Model saved.


In [11]:
def optimize_prompt(text):
    seq = encode_pad(text)
    seq = tf.keras.preprocessing.sequence.pad_sequences([seq], maxlen=X_train.shape[1])
    seq = tf.convert_to_tensor(seq)

    # Load weights for inference
    encoder.load_weights(MODEL_PATH + "_enc.weights.h5")
    decoder.load_weights(MODEL_PATH + "_dec.weights.h5")

    enc_out, h, c = encoder(seq)
    dec_input = tf.constant([[1]])  # <bos>

    result = []

    for _ in range(60):
        logits, h, c = decoder(dec_input, h, c, enc_out)
        pred = tf.argmax(logits[0,0]).numpy()
        if pred == 2:  # <eos>
            break
        result.append(pred)
        dec_input = tf.constant([[pred]])

    return tok.decode(result)

In [12]:
optimize_prompt("hey can u write python code to sort a list but also explain each step and add comments and give time complexity")

'machine learning exam questions.\n- difficulty: hard.\n- provide concise answers.\n- avoid overly long explanations.'

In [13]:
optimize_prompt("pls rewrite this paragraph in simple and professional tone and remove grammar mistakes but dont change meaning ok")

'explain binary search.\n- hindi.\n- हिंदी में सरल और स्पष्ट लिखें।\n- avoid heavy technical terms.'

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd

# =============================================
# CONFIG
# =============================================
TRAIN_PATH = "data/prompts_train.csv"
VAL_PATH   = "data/prompts_val.csv"

MAX_LEN = 128
EMBED = 256
FF_DIM = 512
NUM_HEADS = 4
NUM_LAYERS = 3
BATCH = 16
EPOCHS = 10

MODEL_PATH = "tf_transformer_prompt_optimizer"


# =============================================
# TOKENIZER
# =============================================
tokenizer = tf.keras.preprocessing.text.Tokenizer(filters="", oov_token="<unk>")

def build_tokenizer(texts):
    tokenizer.fit_on_texts(texts)
    tokenizer.word_index["<pad>"] = 0

def encode(text):
    seq = tokenizer.texts_to_sequences([text])[0]
    seq = [1] + seq + [2]   # <bos>=1, <eos>=2
    return seq[:MAX_LEN]

def decode(ids):
    inv = {v:k for k,v in tokenizer.word_index.items()}
    words = [inv.get(i,"") for i in ids if i > 2]
    return " ".join(words)


# =============================================
# LOAD DATA
# =============================================
df_train = pd.read_csv(TRAIN_PATH)
df_val   = pd.read_csv(VAL_PATH)

all_text = df_train["raw_prompt"].tolist() + df_train["optimized_prompt"].tolist()
build_tokenizer(all_text)

VOCAB = len(tokenizer.word_index) + 1
print("Vocabulary size:", VOCAB)


def prepare(df):
    X = [encode(x) for x in df["raw_prompt"]]
    Y = [encode(y) for y in df["optimized_prompt"]]

    X = tf.keras.preprocessing.sequence.pad_sequences(X, maxlen=MAX_LEN, padding="post")
    Y = tf.keras.preprocessing.sequence.pad_sequences(Y, maxlen=MAX_LEN, padding="post")
    return X, Y


X_train, Y_train = prepare(df_train)
X_val, Y_val = prepare(df_val)

train_ds = tf.data.Dataset.from_tensor_slices((X_train, Y_train)).shuffle(4000).batch(BATCH)
val_ds   = tf.data.Dataset.from_tensor_slices((X_val, Y_val)).batch(BATCH)


# =============================================
# POSITIONAL ENCODING
# =============================================
def positional_encoding(max_len, embed_dim):
    pos = np.arange(max_len)[:, np.newaxis]
    i = np.arange(embed_dim)[np.newaxis, :]
    angle_rates = 1 / np.power(10000, (2*(i//2)) / embed_dim)
    angles = pos * angle_rates
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(angles[np.newaxis, ...], dtype=tf.float32)


positional = positional_encoding(MAX_LEN, EMBED)


# =============================================
# TRANSFORMER BLOCK
# =============================================
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation="relu"),
            tf.keras.layers.Dense(embed_dim),
        ])
        self.norm1 = tf.keras.layers.LayerNormalization()
        self.norm2 = tf.keras.layers.LayerNormalization()
        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    def call(self, x, context, training=False):
        attn_output = self.att(x, context)
        out1 = self.norm1(x + attn_output)
        ffn_output = self.ffn(out1)
        out2 = self.norm2(out1 + ffn_output)
        return out2


# =============================================
# BUILD TRANSFORMER MODEL
# =============================================
def build_transformer():
    enc_in = tf.keras.Input((MAX_LEN,))
    dec_in = tf.keras.Input((MAX_LEN,))

    embed = tf.keras.layers.Embedding(VOCAB, EMBED)

    # Encoder
    x = embed(enc_in)
    x += positional

    for _ in range(NUM_LAYERS):
        x = TransformerBlock(EMBED, NUM_HEADS, FF_DIM)(x, x, training=False)

    enc_out = x

    # Decoder
    y = embed(dec_in)
    y += positional

    for _ in range(NUM_LAYERS):
        y = TransformerBlock(EMBED, NUM_HEADS, FF_DIM)(y, enc_out, training=False)

    logits = tf.keras.layers.Dense(VOCAB)(y)

    model = tf.keras.Model([enc_in, dec_in], logits)
    return model


model = build_transformer()
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')
optimizer = tf.keras.optimizers.Adam()


# =============================================
# TRAIN STEP (fixed)
# =============================================
@tf.function
def train_step(src, tgt):
    # Decoder input = whole target
    dec_in = tgt

    # Decoder label = shifted target
    dec_out = tf.concat([tgt[:, 1:], tf.zeros_like(tgt[:, :1])], axis=1)

    with tf.GradientTape() as tape:
        logits = model([src, dec_in], training=True)
        loss_per_token = loss_fn(dec_out, logits)

        # mask out PAD tokens
        mask = tf.cast(tf.not_equal(dec_out, 0), tf.float32)
        loss = tf.reduce_sum(loss_per_token * mask) / tf.reduce_sum(mask)

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss


# =============================================
# TRAIN LOOP
# =============================================
for epoch in range(EPOCHS):
    total = 0
    steps = 0
    for src, tgt in train_ds:
        loss = train_step(src, tgt)
        total += loss
        steps += 1
    print(f"Epoch {epoch+1}/{EPOCHS}  Loss: {total/steps:.4f}")



c:\Users\harih\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Vocabulary size: 263

Epoch 1/10  Loss: 0.7239
Epoch 2/10  Loss: 0.3763
Epoch 3/10  Loss: 0.3693
Epoch 4/10  Loss: 0.3661
Epoch 5/10  Loss: 0.3633
Epoch 6/10  Loss: 0.3614
Epoch 7/10  Loss: 0.3609
Epoch 8/10  Loss: 0.3587
Epoch 9/10  Loss: 0.3587
Epoch 10/10  Loss: 0.3580


ValueError: Invalid filepath extension for saving. Please add either a `.keras` extension for the native Keras format (recommended) or a `.h5` extension. Use `model.export(filepath)` if you want to export a SavedModel for use with TFLite/TFServing/etc. Received: filepath=tf_transformer_prompt_optimizer.

In [4]:
model.save("tf_transformer_prompt_optimizer.h5")

In [5]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [12]:
import tensorflow as tf
import numpy as np
import pickle

# =========================
# CONFIG (same as training)
# =========================
MODEL_PATH = "tf_transformer_prompt_optimizer.h5"
TOKENIZER_PATH = "tokenizer.pkl"

MAX_LEN = 128
EMBED = 256
FF_DIM = 512
NUM_HEADS = 4
NUM_LAYERS = 3


# =========================
# CUSTOM LAYER (same as training)
# =========================
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation="relu"),
            tf.keras.layers.Dense(embed_dim),
        ])

        self.norm1 = tf.keras.layers.LayerNormalization()
        self.norm2 = tf.keras.layers.LayerNormalization()

    def call(self, x, context):
        attn_output = self.att(x, context)
        out1 = self.norm1(x + attn_output)
        ffn_output = self.ffn(out1)
        out2 = self.norm2(out1 + ffn_output)
        return out2


# =========================
# LOAD TOKENIZER
# =========================
with open(TOKENIZER_PATH, "rb") as f:
    tokenizer = pickle.load(f)

VOCAB = len(tokenizer.word_index) + 1


# =========================
# POSITIONAL ENCODING
# =========================
def positional_encoding(max_len, embed_dim):
    pos = np.arange(max_len)[:, None]
    i = np.arange(embed_dim)[None, :]
    angle_rates = 1 / np.power(10000, (2*(i//2)) / embed_dim)
    angles = pos * angle_rates
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(angles[None, ...], tf.float32)

positional = positional_encoding(MAX_LEN, EMBED)


# =========================
# REBUILD MODEL
# =========================
def build_model():

    enc_in = tf.keras.Input((MAX_LEN,))
    dec_in = tf.keras.Input((MAX_LEN,))

    embed = tf.keras.layers.Embedding(VOCAB, EMBED)

    # Encoder
    x = embed(enc_in)
    x = x + positional

    for _ in range(NUM_LAYERS):
        x = TransformerBlock(EMBED, NUM_HEADS, FF_DIM)(x, x)

    enc_out = x

    # Decoder
    y = embed(dec_in)
    y = y + positional

    for _ in range(NUM_LAYERS):
        y = TransformerBlock(EMBED, NUM_HEADS, FF_DIM)(y, enc_out)

    logits = tf.keras.layers.Dense(VOCAB)(y)

    return tf.keras.Model([enc_in, dec_in], logits)


model = build_model()

# ⭐ load weights
model.load_weights(MODEL_PATH)

print("✅ Model rebuilt + weights loaded")


# =========================
# ENCODE / DECODE
# =========================
def encode(text):
    seq = tokenizer.texts_to_sequences([text])[0]
    seq = [1] + seq + [2]
    return seq

def decode(ids):
    inv = {v: k for k, v in tokenizer.word_index.items()}
    return " ".join(inv.get(i, "") for i in ids if i > 2)


# =========================
# INFERENCE
# =========================
def optimize_prompt(text):

    seq = encode(text)
    seq = tf.keras.preprocessing.sequence.pad_sequences([seq], maxlen=MAX_LEN)
    enc_in = tf.convert_to_tensor(seq)

    dec = [1]  # <bos>

    for _ in range(MAX_LEN):

        dec_pad = tf.keras.preprocessing.sequence.pad_sequences([dec], maxlen=MAX_LEN)

        # ⭐ IMPORTANT FIX
        dec_pad = tf.convert_to_tensor(dec_pad)

        logits = model([enc_in, dec_pad], training=False)

        next_id = int(tf.argmax(logits[0, len(dec)-1]))

        if next_id == 2:
            break

        dec.append(next_id)

    return decode(dec)




✅ Model rebuilt + weights loaded
